In [ ]:
import typing

import numpy as np
import matplotlib.pyplot as plt
from jaxtyping import Float, Int
from scipy.optimize import linprog

import muutils.dbg
muutils.dbg.DBG_TENSOR_ARRAY_SUMMARY_DEFAULTS["sparkline_logy"] = True
from muutils.dbg import dbg_tensor
from muutils.tensor_info import array_summary

from attention_motifs.util import plot_figs, load_activations, get_single_attn_pattern
from attention_motifs.transition_tensor import transition_tensor, tt_fig
from attention_motifs.math import compute_envelope_params, linear_plot


In [ ]:
MODEL_NAME: str = "pythia-14m"
MODEL_CFG: dict
PROMPTS: list[dict]
ACTIVATIONS: list[np.lib.npyio.NpzFile]
MODEL_CFG, PROMPTS, ACTIVATIONS = load_activations(MODEL_NAME)

In [ ]:
_idxs, _tt, _res = transition_tensor(
	get_single_attn_pattern(0, 0, 0, activations=ACTIVATIONS),
	exact=20,
	approx_l10=4.0,
	approx_pts=20,
)
dbg_tensor(_idxs)
dbg_tensor(_tt)
dbg_tensor(_res)

print()

In [ ]:



def fig(
	A: np.ndarray,
	axs: list[plt.Axes],
	p_idx: int,
	lyr: int,
	head: int,
	p_threshold: float = 0.95,
):
	n_ctx: int = A.shape[0]
	#
	axs[0].set_title("raw attention")
	axs[0].matshow(A)

	# compute transition_tensor
	idxs, tt, res = transition_tensor(A, exact=100, approx_l10=7.0, approx_pts=100)

	#
	axs[1].set_title("transition tensor")
	# axs[1].matshow(np.log10(1 - tt[:, :, 0].T + 1e-8), aspect=tt.shape[0] / tt.shape[1])
	# use the +1 log
	axs[1].matshow(np.log1p(1 - tt[:, :, 0].T), aspect=(tt.shape[0] / tt.shape[1]))
	axs[1].set_xticks(range(len(idxs)))
	axs[1].set_xticklabels(idxs)
	axs[1].tick_params(axis="x", rotation=90)

	#
	# axs[2].set_title("residuals tensor")
	# # aspect should be such that the image is square, although the matrix is not
	# axs[2].matshow(res.T, aspect=(res.shape[0] / res.shape[1]))
	# axs[2].set_xticks(range(len(idxs)))
	# axs[2].set_xticklabels(idxs)
	# axs[2].tick_params(axis="x", rotation=90)

	# 
	# axs[3].set_title("residuals tensor (log10)")
	# axs[3].matshow(np.log10(res.T + 1e-8), aspect=(res.shape[0] / res.shape[1]))
	# axs[3].set_xticks(range(len(idxs)))
	# axs[3].set_xticklabels(idxs)
	# axs[3].tick_params(axis="x", rotation=90)

	axs[2].set_title(f"time to transition probability > {p_threshold}")
	indices_raw = np.apply_along_axis(
		lambda row: np.searchsorted(row, p_threshold, side="right"), axis=0, arr=tt[:, :, 0]
	)
	axs[1].plot(indices_raw, np.arange(indices_raw.shape[0]), "r.")
	idxs_with_inf = np.concatenate((idxs, [1e10]))
	indices_adjusted = np.array(idxs_with_inf[indices_raw], dtype=float)
	# if last element, set to inf
	# indices_adjusted[indices_raw == len(idxs)] = 1e10
	indices_adjusted_l10 = np.log10(indices_adjusted[1:])
	axs[2].plot(indices_adjusted_l10, "ro")
	axs[2].set_xlabel("token idx")
	axs[2].set_ylabel("log10(iters to transition)")
	# axs[4].set_yscale("log")
	dbg_tensor(indices_adjusted)
	dbg_tensor(indices_adjusted_l10)
	idxs_x = np.arange(len(indices_adjusted_l10))	
	for envtype in ("lower", "upper", "bestfit"):
		env_lower = compute_envelope_params(
			x=idxs_x,
			y=indices_adjusted_l10,
			envelope_type=envtype,
		)
		axs[2].plot(
			idxs_x,
			linear_plot(idxs_x, env_lower[0], env_lower[1]),
			label=f"{envtype}, $R^2={env_lower[2]:.3f}$",
		)
	axs[2].legend()

	indices_adjusted_diff = np.diff(indices_adjusted)
	axs[3].set_title(f"dist of transition times diff\n${array_summary(indices_adjusted_diff, fmt='latex', dtype=False)}$")
	axs[3].hist(indices_adjusted_diff, bins=10)
	axs[3].set_ylabel("count")

	# 
	# axs[5].set_title("transition time fft")
	# tt_time_fft = np.fft.fft(idxs_with_inf)
	# # print(tt_time_fft)
	# axs[5].plot(np.abs(tt_time_fft), "o-", label="abs")
	# # axs[5].plot(np.angle(tt_time_fft), label="angle")
	# # axs[5].plot(tt_time_fft.real, label="real")
	# # axs[5].plot(tt_time_fft.imag, label="imaginary")
	# axs[5].set_yscale("log")
	# axs[5].legend()



plot_figs(
	n_figures=4,
	model_cfg=MODEL_CFG,
	prompt_dicts=PROMPTS,
	activations=ACTIVATIONS,
	figure_func=fig,
	prompts=[0, 1, 2],
	layers=None,
	heads=[0],
)

plt.show()